# Week 51, Day 2: Bellman–Ford

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w051d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
edges = [("A", "B", 2), ("A", "C", 3), ("C", "B", -2), ("B", "D", 1)]
dist = {"A": 0, "B": float("inf"), "C": float("inf"), "D": float("inf")}

for round_number in range(1, 4):
    for u, v, w in edges:
        if dist[u] + w < dist[v]:
            dist[v] = dist[u] + w
    print(round_number, dist)

In [ ]:
def bellman_ford(vertices, edges, start):
    dist = {v: float("inf") for v in vertices}
    dist[start] = 0
    for _ in range(len(vertices) - 1):
        changed = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                changed = True
        if not changed:
            break
    return dist

edges = [("S", "A", 4), ("S", "B", 5), ("A", "C", -3), ("B", "A", -2), ("C", "D", 2), ("B", "D", 6)]
print(bellman_ford("SABCD", edges, "S"))

In [ ]:
def find_negative_cycle(vertices, edges, start):
    dist = {v: float("inf") for v in vertices}
    parent = {v: None for v in vertices}
    dist[start] = 0
    last = None
    for _ in range(len(vertices)):              # V rounds: the last one only to detect
        last = None
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v], parent[v] = dist[u] + w, u
                last = v
    if last is None:
        return None
    for _ in range(len(vertices)):              # step back onto the cycle
        last = parent[last]
    cycle, v = [last], parent[last]
    while v != last:
        cycle.append(v)
        v = parent[v]
    return cycle[::-1]

edges = [("S", "A", 1), ("A", "B", 1), ("B", "C", -3), ("C", "A", 1), ("C", "D", 2)]
print(find_negative_cycle("SABCD", edges, "S"))
print(find_negative_cycle("SABCD", [("S", "A", 1), ("A", "B", 1), ("B", "C", -1), ("C", "A", 1)], "S"))

In [ ]:
def cheapest_within(flights, start, goal, max_flights):
    best = {start: 0}
    for _ in range(max_flights):
        before = dict(best)                     # this round builds on the last one only
        for u, v, price in flights:
            if u in before and before[u] + price < best.get(v, float("inf")):
                best[v] = before[u] + price
    return best.get(goal)

flights = [("LOS", "ACC", 120), ("ACC", "NBO", 300), ("LOS", "ADD", 250), ("ADD", "NBO", 100), ("LOS", "NBO", 600)]
for k in (1, 2):
    print(k, "flight(s):", cheapest_within(flights, "LOS", "NBO", k))

In [ ]:
import math

rates = {("NGN", "USD"): 0.00065, ("USD", "EUR"): 0.92, ("EUR", "NGN"): 1700.0,
         ("USD", "NGN"): 1530.0, ("EUR", "USD"): 1.08}
loop = [("NGN", "USD"), ("USD", "EUR"), ("EUR", "NGN")]
product = math.prod(rates[e] for e in loop)
log_sum = sum(-math.log(rates[e]) for e in loop)
print(round(product, 4), round(log_sum, 4), "arbitrage!" if log_sum < -1e-9 else "no gain")

In [ ]:
def solve(variables, constraints):
    """constraints: (u, v, c) meaning x[v] - x[u] <= c."""
    edges = [("*", v, 0) for v in variables] + list(constraints)
    dist = {v: 0 for v in variables}
    dist["*"] = 0
    for _ in range(len(variables)):
        for u, v, c in edges:
            if dist[u] + c < dist[v]:
                dist[v] = dist[u] + c
    if any(dist[u] + c < dist[v] for u, v, c in edges):
        return None
    shift = -min(dist[v] for v in variables)           # make the earliest day 0
    return {v: dist[v] + shift for v in variables}

# builder starts first; plumber at least 2 days after builder; painter at least 3 after plumber,
# and the painter at most 4 days after the builder
cons = [("plumber", "builder", -2), ("painter", "plumber", -3), ("builder", "painter", 4)]
print(solve(["builder", "plumber", "painter"], cons))
print(solve(["builder", "plumber", "painter"], [("plumber", "builder", -2), ("painter", "plumber", -3), ("builder", "painter", 4), ("builder", "painter", 1)]))

> **Key idea.** Bellman-Ford relaxes every edge in rounds; after round k, all shortest paths of at most k edges are found, so V − 1 rounds suffice: O(V × E), correct with negative weights, and it can stop early when a round changes nothing. If round V still relaxes an edge, a negative cycle is reachable; parent links followed V times lead onto it, and everything reachable from it has distance minus infinity. Copying distances each round limits paths to k edges. Logarithms turn products into sums, so arbitrage is a negative cycle in −log(rate). Difference constraints x_v − x_u ≤ c are edges u → v of weight c; shortest distances satisfy them, and a negative cycle means they contradict each other.

### Exercise: Bellman-Ford

Write `bellman_ford(vertices, edges, start)`: a dict of shortest distances (infinity for unreachable vertices), stopping early when a round changes nothing; raise `ValueError` if a negative cycle is reachable from `start`. Then `rounds_needed(vertices, edges, start)`: how many rounds actually changed something.

In [ ]:
def bellman_ford(vertices, edges, start):
    ___

def rounds_needed(vertices, edges, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef slow(vertices, edges, s):\n    best = {v: float(\"inf\") for v in vertices}\n    best[s] = 0\n    adj = {}\n    for u, v, w in edges:\n        adj.setdefault(u, []).append((v, w))\n    def walk(v, d, seen):\n        if d < best[v]:\n            best[v] = d\n        for x, w in adj.get(v, []):\n            if x not in seen:\n                walk(x, d + w, seen | {x})\n    walk(s, 0, {s})\n    return best\ndef test_bf():\n    \"distances\"\n    edges = [(\"S\", \"A\", 4), (\"S\", \"B\", 5), (\"A\", \"C\", -3), (\"B\", \"A\", -2), (\"C\", \"D\", 2), (\"B\", \"D\", 6)]\n    assert bellman_ford(\"SABCD\", edges, \"S\") == {\"S\": 0, \"A\": 3, \"B\": 5, \"C\": 0, \"D\": 2}\n    assert bellman_ford(\"AB\", [], \"A\") == {\"A\": 0, \"B\": float(\"inf\")}\n    try:\n        bellman_ford(\"SAB\", [(\"S\", \"A\", 1), (\"A\", \"B\", -2), (\"B\", \"A\", 1)], \"S\")\n        assert False\n    except ValueError:\n        pass\n    assert bellman_ford(\"SAB\", [(\"A\", \"B\", -2), (\"B\", \"A\", 1)], \"S\")[\"A\"] == float(\"inf\"), \"unreachable cycles don't matter\"\n    rng = random.Random(1)\n    for _ in range(100):\n        vs = list(range(rng.randint(1, 6)))\n        perm = rng.sample(vs, len(vs))\n        edges = [(perm[i], perm[j], rng.randint(-5, 9)) for i in range(len(vs)) for j in range(i + 1, len(vs)) if rng.random() < 0.5]\n        assert bellman_ford(vs, edges, perm[0]) == slow(vs, edges, perm[0])\n    backwards = [(\"C\", \"D\", 1), (\"B\", \"C\", 1), (\"A\", \"B\", 1), (\"S\", \"A\", 1)]\n    assert rounds_needed(\"SABCD\", backwards, \"S\") == 4 and rounds_needed(\"SABCD\", backwards[::-1], \"S\") == 1")

### Exercise: Finding the cycle

Write `negative_cycle(vertices, edges)`: one negative cycle **anywhere** in the graph (not only reachable from some start) as a list `[v1, ..., vk]` with an edge from each to the next and from vk to v1, or `None`. Start every distance at 0, which is the same as adding a start vertex with a 0-weight edge to every vertex. Then `unbounded(vertices, edges, start)`: the sorted list of vertices whose distance from `start` is minus infinity.

In [ ]:
def negative_cycle(vertices, edges):
    ___

def unbounded(vertices, edges, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef weight(edges, c):\n    best = {}\n    for u, v, w in edges:\n        best[(u, v)] = min(w, best.get((u, v), float(\"inf\")))\n    return sum(best[(c[i], c[(i + 1) % len(c)])] for i in range(len(c)))\ndef test_cycle():\n    \"negative cycles\"\n    edges = [(\"S\", \"A\", 1), (\"A\", \"B\", 1), (\"B\", \"C\", -3), (\"C\", \"A\", 1), (\"C\", \"D\", 2), (\"E\", \"S\", 1)]\n    c = negative_cycle(\"SABCDE\", edges)\n    assert sorted(c) == [\"A\", \"B\", \"C\"] and weight(edges, c) < 0\n    assert unbounded(\"SABCDE\", edges, \"S\") == [\"A\", \"B\", \"C\", \"D\"] and unbounded(\"SABCDE\", edges, \"D\") == []\n    assert negative_cycle(\"AB\", [(\"A\", \"B\", 1), (\"B\", \"A\", -1)]) is None\n    assert negative_cycle(\"X\", [(\"X\", \"X\", -1)]) == [\"X\"]\n    rng = random.Random(2)\n    for _ in range(200):\n        vs = list(range(rng.randint(1, 5)))\n        edges = [(rng.choice(vs), rng.choice(vs), rng.randint(-4, 6)) for _ in range(rng.randint(0, 8))]\n        edges = [e for e in edges if e[0] != e[1] or rng.random() < 0.3]\n        c = negative_cycle(vs, edges)\n        if c is not None:\n            assert len(set(c)) == len(c) and weight(edges, c) < 0")

### Exercise: With at most k stops

Write `cheapest_flight(flights, start, goal, max_stops)`: the cheapest price from start to goal using at most `max_stops` intermediate airports (so at most `max_stops + 1` flights), or `None`. Copy the prices each round. Then `price_by_stops(flights, start, goal, max_stops)`: a list of the cheapest price allowing 0, 1, ..., max_stops stops (`None` where impossible).

In [ ]:
def cheapest_flight(flights, start, goal, max_stops):
    ___

def price_by_stops(flights, start, goal, max_stops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef test_flights():\n    \"flights\"\n    flights = [(\"LOS\", \"ACC\", 120), (\"ACC\", \"NBO\", 300), (\"LOS\", \"ADD\", 250), (\"ADD\", \"NBO\", 100), (\"LOS\", \"NBO\", 600)]\n    assert cheapest_flight(flights, \"LOS\", \"NBO\", 0) == 600 and cheapest_flight(flights, \"LOS\", \"NBO\", 1) == 350\n    assert cheapest_flight(flights, \"NBO\", \"LOS\", 3) is None and price_by_stops(flights, \"LOS\", \"NBO\", 2) == [600, 350, 350]\n    chain = [(\"A\", \"B\", 1), (\"B\", \"C\", 1), (\"C\", \"D\", 1), (\"A\", \"D\", 10)]\n    assert price_by_stops(chain, \"A\", \"D\", 2) == [10, 10, 3]\n    rng = random.Random(3)\n    for _ in range(100):\n        names = \"abcde\"\n        fl = [(a, b, rng.randint(1, 20)) for a in names for b in names if a != b and rng.random() < 0.4]\n        k = rng.randint(0, 3)\n        best = None\n        for L in range(0, k + 1):\n            for mid in itertools.permutations(\"bcd\", L):\n                route = (\"a\",) + mid + (\"e\",)\n                price = 0\n                for x, y in zip(route, route[1:]):\n                    ps = [p for u, v, p in fl if u == x and v == y]\n                    if not ps:\n                        price = None\n                        break\n                    price += min(ps)\n                if price is not None:\n                    best = price if best is None else min(best, price)\n        assert cheapest_flight(fl, \"a\", \"e\", k) == best")

### Exercise: Free money?

`rates` maps `(from, to)` currency pairs to exchange rates. Write `arbitrage(rates)`: a loop of currencies `[c1, c2, ..., ck]` whose rates multiply to more than 1 (by at least one part in a billion), found as a negative cycle with weights −log(rate), or `None`. Then `gain(rates, loop)`: the product of the rates round the loop (including back to the start).

In [ ]:
import math

def arbitrage(rates):
    ___

def gain(rates, loop):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\ndef test_arbitrage():\n    \"arbitrage\"\n    rates = {(\"NGN\", \"USD\"): 0.00065, (\"USD\", \"EUR\"): 0.92, (\"EUR\", \"NGN\"): 1700.0, (\"USD\", \"NGN\"): 1530.0, (\"EUR\", \"USD\"): 1.08}\n    loop = arbitrage(rates)\n    assert loop is not None and gain(rates, loop) > 1\n    fair = {(\"A\", \"B\"): 2.0, (\"B\", \"A\"): 0.5, (\"B\", \"C\"): 3.0, (\"C\", \"B\"): 1 / 3, (\"A\", \"C\"): 6.0, (\"C\", \"A\"): 1 / 6}\n    assert arbitrage(fair) is None and abs(gain(fair, [\"A\", \"B\", \"C\"]) - 1) < 1e-12\n    assert gain({(\"x\", \"y\"): 2.0, (\"y\", \"x\"): 0.75}, [\"x\", \"y\"]) == 1.5")

### Exercise: Scheduling with constraints

Constraints arrive as text lines, `A after B by at least 3` (x_A − x_B ≥ 3) or `A after B by at most 5` (x_A − x_B ≤ 5). Write `schedule(lines)`: the day each task starts, with the earliest task on day 0, satisfying every line, or `None` if they contradict each other. Turn each line into a difference constraint and use Bellman-Ford.

In [ ]:
def schedule(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def ok(days, lines):\n    for line in lines:\n        a, _, b, _, _, kind, n = line.split()\n        diff = days[a] - days[b]\n        if kind == \"least\" and diff < int(n) or kind == \"most\" and diff > int(n):\n            return False\n    return True\ndef test_schedule():\n    \"schedules\"\n    lines = [\"plumber after builder by at least 2\", \"painter after plumber by at least 3\", \"painter after builder by at most 6\"]\n    days = schedule(lines)\n    assert ok(days, lines) and min(days.values()) == 0 and days[\"builder\"] == 0\n    assert schedule(lines + [\"painter after builder by at most 4\"]) is None\n    assert schedule([]) == {}\n    lines = [\"b after a by at least 1\", \"c after b by at least 1\", \"a after c by at least -5\", \"d after a by at most 0\", \"a after d by at most 0\"]\n    days = schedule(lines)\n    assert ok(days, lines) and days[\"a\"] == days[\"d\"]")

### Bug hunt: One round isn't enough

This relaxes every edge only **once**, which finds shortest paths only when edges happen to be listed in a helpful order. Here the edges are listed backwards, so after one round D is still unreached: this prints `inf`. Do V − 1 rounds so it prints `3`.

In [ ]:
vertices = "ABCD"
edges = [("C", "D", 1), ("B", "C", 1), ("A", "B", 1)]
dist = {v: float("inf") for v in vertices}
dist["A"] = 0
for u, v, w in edges:
    if dist[u] + w < dist[v]:
        dist[v] = dist[u] + w
print(dist["D"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"enough rounds\"\n    expect_output(\"3\")")

### Bug hunt: Too many flights in one round

This "at most one flight" search updates prices in place, so within the same round a price just found for ACC is used to reach NBO: it chains two flights and prints `420`. Copy the prices at the start of each round, so it prints `600`.

In [ ]:
flights = [("LOS", "ACC", 120), ("ACC", "NBO", 300), ("LOS", "NBO", 600)]
best = {"LOS": 0}
for _ in range(1):
    for u, v, price in flights:
        if u in best and best[u] + price < best.get(v, float("inf")):
            best[v] = best[u] + price
print(best["NBO"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one round, one flight\"\n    expect_output(\"600\")")

### Bug hunt: Unreachable isn't zero

This uses `dist.get(u, 0)`, treating vertices it hasn't reached as distance 0, so an edge out of an unreachable vertex looks like a short cut: X can't be reached from A, yet this prints `-5` for B. Treat unreached vertices as infinitely far so it prints `2`.

In [ ]:
edges = [("A", "B", 2), ("X", "B", -5)]
dist = {"A": 0}
for _ in range(2):
    for u, v, w in edges:
        if dist.get(u, 0) + w < dist.get(v, float("inf")):
            dist[v] = dist.get(u, 0) + w
print(dist["B"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"infinity for unreached\"\n    expect_output(\"2\")")

### Puzzle: Bellman-Ford

Arrange the lines into Bellman-Ford with early stopping. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Edges in this order: B->C 2, A->B 5, A->C 9, C->D -4, S->A 1. Start at S.

p = ___   # dist after round 1, as a dict (use float("inf") for unreached)
q = ___   # the final distance to D
r = ___   # the most rounds Bellman-Ford needs on 5 vertices without early stopping
s = ___   # True or False: a cycle of weights 2, -1, -1 is a negative cycle
t = ___   # the weight to use for an exchange rate r when looking for arbitrage, as a string like "-log(r)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    inf = float(\"inf\")\n    assert p == {\"S\": 0, \"A\": 1, \"B\": inf, \"C\": inf, \"D\": inf} and q == 4 and r == 4, \"p, q and r\"\n    assert s is False and t.replace(\" \", \"\") == \"-log(r)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: all pairs with Johnson's reweighting

To get shortest distances between **every** pair of vertices in a sparse graph with some negative edges (but no negative cycle), running Bellman-Ford from every vertex costs O(V² × E). Donald Johnson's 1977 algorithm does better: run Bellman-Ford **once**, then Dijkstra from every vertex, O(V × E log V).

1. Add a new vertex q with a 0-weight edge to every vertex, and run Bellman-Ford from q to get h(v) for every v (raise `ValueError` if there's a negative cycle).
2. **Reweight** each edge: w'(u, v) = w(u, v) + h(u) − h(v). Because h(v) ≤ h(u) + w(u, v), every new weight is **non-negative**. And along any path from s to t the h terms cancel in pairs, so every path's new length is its old length plus h(s) − h(t): the same paths are shortest.
3. Run Dijkstra from each vertex with the new weights, and convert back: d(s, t) = d'(s, t) − h(s) + h(t).

Write `johnson(vertices, edges)`: a dict of dicts, `result[s][t]` the shortest distance (only for t reachable from s, including s itself at 0). Also `reweighted(vertices, edges)`: the list of `(u, v, new_weight)` triples, all non-negative. The checks compare with Bellman-Ford from every vertex.

In [ ]:
import heapq

def reweighted(vertices, edges):
    ___

def johnson(vertices, edges):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef bf(vertices, edges, s):\n    dist = {v: float(\"inf\") for v in vertices}\n    dist[s] = 0\n    for _ in range(len(vertices)):\n        for u, v, w in edges:\n            if dist[u] + w < dist[v]:\n                dist[v] = dist[u] + w\n    return {v: d for v, d in dist.items() if d < float(\"inf\")}\ndef test_johnson():\n    \"all pairs\"\n    edges = [(\"S\", \"A\", 4), (\"S\", \"B\", 5), (\"A\", \"C\", -3), (\"B\", \"A\", -2), (\"C\", \"D\", 2), (\"B\", \"D\", 6)]\n    r = johnson(\"SABCD\", edges)\n    assert r[\"S\"] == {\"S\": 0, \"A\": 3, \"B\": 5, \"C\": 0, \"D\": 2} and r[\"D\"] == {\"D\": 0} and r[\"B\"][\"C\"] == -5\n    assert all(w >= 0 for _, _, w in reweighted(\"SABCD\", edges))\n    try:\n        johnson(\"AB\", [(\"A\", \"B\", 1), (\"B\", \"A\", -2)])\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(4)\n    for _ in range(80):\n        vs = list(range(rng.randint(1, 7)))\n        perm = rng.sample(vs, len(vs))\n        edges = [(perm[i], perm[j], rng.randint(-5, 9)) for i in range(len(vs)) for j in range(len(vs)) if i < j and rng.random() < 0.4]\n        edges += [(perm[j], perm[i], rng.randint(6, 12)) for i in range(len(vs)) for j in range(len(vs)) if i < j and rng.random() < 0.2]\n        try:\n            want = {s: bf(vs, edges, s) for s in vs}\n            neg = any(dist[u] + w < dist[v] for s in vs for dist in [want[s]] for u, v, w in edges if u in dist and v in dist)\n        except Exception:\n            neg = True\n        if neg:\n            continue\n        assert johnson(vs, edges) == want\n        assert all(w >= 0 for _, _, w in reweighted(vs, edges))")

## Warm-up

Yesterday's graph that fooled Dijkstra's algorithm: B ends at 1 and D at 2, the true distances. Each round relaxes every edge; by the second round nothing changes any more.

## Relax everything, repeatedly

Bellman-Ford keeps the same tentative distances as Dijkstra's algorithm (0 at the start, infinity elsewhere), but instead of choosing which vertex to process next, it does V − 1 **rounds**, and each round relaxes **every edge**.

Why V − 1? A shortest path (with no negative cycle around) never repeats a vertex, so it has at most V − 1 edges. After round 1, every shortest path of **one** edge has been found; after round k, every shortest path of at most k edges, because its last edge gets relaxed in round k with the correct distance already at its start. So after V − 1 rounds, all shortest distances are right, whatever the order of the edges and whatever their signs.

Each round costs O(E), so the algorithm is **O(V × E)**: slower than Dijkstra's O((V + E) log V), but simpler, and correct with negative weights. Two practical touches:

- **Stop early** if a whole round changes nothing: nothing will ever change again. On many graphs this finishes long before V − 1 rounds.
- An **edge list** suits it best, since every round walks every edge.

The algorithm is named after Richard Bellman (1958) and Lester Ford Jr. (1956); Edward Moore published it too (1959).

`float("inf")` makes unreachable vertices easy: infinity plus any finite weight is still infinity, so it never "improves" anything.

## Negative cycles

If a cycle's weights add up to less than zero, going round it once more always makes a path shorter, so vertices reachable from it have no shortest distance (it's minus infinity). Bellman-Ford detects this with one extra round: after V − 1 rounds all true distances are final, so **if any edge can still be relaxed in round V, there's a negative cycle** reachable from the start.

To **find** the cycle: record parents as you relax. Take a vertex v relaxed in round V and follow parents back V times; that lands you on the cycle (a path of V parent steps must repeat a vertex). Then follow parents from there until you come back round.

To find every vertex whose distance is minus infinity: everything **reachable** from a vertex relaxed in round V. A BFS from those vertices marks them all.

The cycle A → B → C → A weighs 1 + (−3) + 1 = −1. Change −3 to −1 and the cycle weighs +1, which is harmless.

## At most k edges

"The cheapest flight with at most k stops" limits the **number of edges**. Bellman-Ford's rounds are exactly that: after round i, distances are the best using at most i edges, **provided each round only builds on the previous round's values**. In-place updates (as above) can chain several edges in one round, which is fine for ordinary shortest paths but wrong here. So copy the distances at the start of each round, and relax from the copy. k stops means k + 1 flights, so k + 1 rounds.

## Rates and logarithms

Some problems multiply along a path instead of adding: exchange rates, probabilities, growth factors. Logarithms turn products into sums (log(ab) = log a + log b), so a product along a path becomes a sum of logs, and shortest-path algorithms apply.

**Currency arbitrage** is the classic case: if converting round a loop of currencies multiplies your money by more than 1, there's free money. With edge weights −log(rate), "product greater than 1" becomes "sum less than 0": a **negative cycle**. Floating-point rounding can make a product of exactly 1 look slightly bigger, so compare with a small tolerance.

## Difference constraints

A system of constraints like "the plumber must start at least 2 days after the builder" or "the review is at most 5 days after the draft" can be written as **x_v − x_u ≤ c**. Each one is an edge u → v of weight c: in a shortest-path solution, dist[v] ≤ dist[u] + c is exactly the constraint. Add a start vertex with a 0-weight edge to every variable, run Bellman-Ford, and the distances **satisfy every constraint**; if there's a negative cycle, the constraints contradict each other and no solution exists. (Weights here are often negative, which is why Bellman-Ford, not Dijkstra, is the tool.)

"At least 2 days after" is x_plumber − x_builder ≥ 2, which is x_builder − x_plumber ≤ −2: an edge plumber → builder of weight −2.

## Practice

## Quiz

## Daily challenge

## Before you go

Find three real exchange rates between currencies and check whether converting round the loop gains or loses (it will lose a little, which is the fee). Tomorrow: more union-find, going further than last week, as preparation for minimum spanning trees.